# Week 01 — Rolling Beta Instability\n\nThis notebook estimates rolling market sensitivity using public tickers and 63-day rolling OLS.\n

In [ ]:
import sys\nfrom pathlib import Path\n\nROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nsys.path.append(str(ROOT / 'src'))\n\nfrom quant_research.data import download_prices\nfrom quant_research.returns import simple_returns\nfrom quant_research.rolling_models import rolling_ols_alpha_beta_r2\nfrom quant_research.risk_metrics import drawdown\n\nTICKERS = ['SPY', 'QQQ', 'TLT', 'GLD', 'HYG', 'SHY', '^VIX']\nBENCHMARK = 'SPY'\nASSETS = ['QQQ', 'TLT', 'GLD', 'HYG', 'SHY']\nWINDOW = 63\nSTART_DATE = '2020-01-01'\n

In [ ]:
prices = download_prices(TICKERS, start=START_DATE, cache_path=ROOT / 'data/raw/week01_prices.csv')\nreturns = simple_returns(prices)\nprices.tail()\n

In [ ]:
rolling_stats = {\n    asset: rolling_ols_alpha_beta_r2(returns[asset], returns[BENCHMARK], window=WINDOW)\n    for asset in ASSETS\n}\n\nsummary = []\nfor asset, stats in rolling_stats.items():\n    summary.append({\n        'asset': asset,\n        'latest_beta': stats['beta'].iloc[-1],\n        'latest_r_squared': stats['r_squared'].iloc[-1],\n        'avg_beta': stats['beta'].mean(),\n        'max_beta': stats['beta'].max(),\n        'min_beta': stats['beta'].min(),\n        'avg_residual_vol_daily': stats['residual_vol_daily'].mean(),\n    })\n\nimport pandas as pd\npd.DataFrame(summary).set_index('asset').round(4)\n

## Interpretation\n\n- Rising beta + rising R² means the asset is becoming more market-driven.\n- Rising beta during stress means diversification may be weaker when needed most.\n- Unstable beta makes static hedge ratios less reliable.\n